## England

## Integrated Care Boards / Sub-ICB Locations (England, April 2026)

**Note on count discrepancy:** the "42" figure often quoted for ICBs (including in secondary sources found while researching this) is the 2022-2026 baseline, not the current count. Confirmed directly from NHS England's own page: the Integrated Care Boards (Establishment and Abolition) Order 2026 abolished 12 existing ICBs and established 6 new ones (net -6) with effect from 1 April 2026 -- covering only London, East of England, and South East regions in this first phase. NHS England explicitly references "the 36 ICBs" post-reorganisation. A second phase of mergers is expected April 2027. Correct current count: **36**, not 42.

**Note on missing full-resolution boundary:** no BFC (full resolution) release could be found for the April 2026 ICB *parent* tier specifically -- only BSC (200m generalised) turned up across multiple searches (data.gov.uk, ckan, geoportal). This may simply be a newer vintage that hasn't had all five resolution variants published yet, since ICB boundaries don't follow the standard May/December ONS release cycle.

**Resolution:** rather than use the coarser BSC parent layer, used the **Sub-ICB Locations (April 2026) EN BFC** layer instead, which carries its parent ICB as an attribute (`ICB26CD`/`ICB26NM`). This gets the England side at full BFC resolution without needing a parent-tier pull at all -- anyone wanting ICB-level boundaries can `dissolve(by="icb_code")` on this layer on demand, since sub-ICBs exactly partition their parent ICB (no approximation in the dissolve).

**Note on schema:** original columns were
`['geometry', 'FID', 'SICBL26CD', 'SICBL26NM', 'ICB26CD', 'ICB26NM', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length', 'GlobalID']`
-- matches the equivalent ONS lookup table's field-naming convention exactly. No parent NHS England Region field in the boundary layer itself (that only exists in a separate lookup table).

**Note on file structure:** saved into a shared `health_boards.gpkg`covering all four nations' health-board-equivalent geographies, with country-suffixed layer names, rather than one file per country/tier:


In [ ]:
import requests
import geopandas as gpd
import pandas as pd
import json
import time

In [ ]:


def fetch_arcgis_layer(base_url, page_size=200, timeout=120, max_retries=3):
    offset = 0
    frames = []
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        for attempt in range(max_retries):
            try:
                r = requests.get(f"{base_url}/query", params=params, timeout=timeout)
                r.raise_for_status()
                data = r.json()
                break
            except (requests.exceptions.RequestException, requests.exceptions.ChunkedEncodingError) as e:
                if attempt == max_retries - 1:
                    raise
                wait = 5 * (attempt + 1)
                print(f"Page at offset {offset}, attempt {attempt+1} failed ({type(e).__name__}), retrying in {wait}s...")
                time.sleep(wait)

        if not data.get("features"):
            break
        frames.append(gpd.GeoDataFrame.from_features(data["features"]))
        print(f"Fetched {offset + len(data['features'])} features so far...")
        if len(data["features"]) < page_size:
            break
        offset += page_size
    return pd.concat(frames, ignore_index=True)

In [3]:
sub_icb = fetch_arcgis_layer(
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/"
    "Sub_Integrated_Care_Board_Locations_April_2026_Boundaries_EN_BFC/FeatureServer/0", 
    page_size=200
)
print(len(sub_icb))
print(sub_icb.columns)

Fetched 106 features so far...
106
Index(['geometry', 'FID', 'SICBL26CD', 'SICBL26NM', 'ICB26CD', 'ICB26NM',
       'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length',
       'GlobalID'],
      dtype='str')


In [4]:
sub_icb = sub_icb.drop(columns=["FID", "Shape__Area", "Shape__Length"])

column_rename = {
    "SICBL26CD": "sub_icb_code",
    "SICBL26NM": "sub_icb_name",
    "ICB26CD":   "icb_code",
    "ICB26NM":   "icb_name",
    "BNG_E":     "easting_bng",
    "BNG_N":     "northing_bng",
    "LONG":      "longitude",
    "LAT":       "latitude",
    "GlobalID":  "global_id",
}

sub_icb = sub_icb.rename(columns=column_rename)
sub_icb.head()

,geometry,sub_icb_code,sub_icb_name,icb_code,icb_name,easting_bng,northing_bng,longitude,latitude,global_id
0,"POLYGON ((-1.34865 53.58333, -1.34844 53.58326...",E38000006,NHS South Yorkshire ICB - 02P,E54000061,NHS South Yorkshire Integrated Care Board,429979,403330,-1.549258,53.52580,c2999ae2-96c7-45ee-a54a-70163684cfdd
1,"MULTIPOLYGON (((0.54225 51.53445, 0.54214 51.5...",E38000007,NHS Essex ICB - 99E,E54000066,NHS Essex Integrated Care Board,564014,194421,0.368068,51.62471,c67f6be7-c638-45c8-99af-da6f781bcbcf
2,"MULTIPOLYGON (((-0.77186 53.2559, -0.77208 53....",E38000008,NHS Nottingham and Nottinghamshire ICB - 02Q,E54000060,NHS Nottingham and Nottinghamshire Integrated ...,468073,384833,-0.978701,53.35603,1ea33093-3238-4643-ad69-37f9babd7d19
3,"POLYGON ((-2.37124 53.66708, -2.37074 53.6661,...",E38000014,NHS Lancashire and South Cumbria ICB - 00Q,E54000048,NHS Lancashire and South Cumbria Integrated Ca...,369490,422806,-2.463604,53.70081,f7b0fe0d-22bc-469d-9145-2dd85528f033
4,"POLYGON ((-3.0568 53.77655, -3.05711 53.77652,...",E38000015,NHS Lancashire and South Cumbria ICB - 00R,E54000048,NHS Lancashire and South Cumbria Integrated Ca...,332817,436634,-3.022025,53.82163,fb79a9b4-34bf-4bc0-a202-10d4751c7d6e


In [5]:
icb_path = r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\processed\health_boards.gpkg"

sub_icb.to_file(icb_path, driver="GPKG", layer="sub_icb_apr2026_EN", index=False)

c:\Users\spspa\miniforge3\envs\raster_env\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


## Wales

## Local Health Boards (Wales, Post-April 2019)

**Note:** different platform from ONS ArcGIS -- this is **GeoNode/GeoServer**
(`datamap.gov.wales`), fetched via a **WFS GetFeature** request rather than
the ArcGIS REST API used for the other layers.


**Note on vintage:** This file, while from 2019, reflects the last actual LHB boundary change (Bridgend moved from
Abertawe Bro Morgannwg to Cwm Taf, forming Cwm Taf Morgannwg UHB; the remainder became Swansea Bay UHB). Wales still has 7 LHBs, unchanged since.

**Note on resolution naming:** this platform uses "High Water Mark" / "Low Water Mark" instead of ONS's BFC/BFE convention. High Water Mark (`lhb_high_water_mark_post_apr_2019`) is the equivalent of BFC (clipped to coastline) -- used here for consistency with the rest of the catalogue. 

**Note on schema:** confirmed 7 features. Two code fields, not self-explanatory from field names alone -- checked actual values before deciding which was primary:
- `area_code` (e.g. `W11000028`) -- ONS-style GSS code, same family as   `PCON24CD`/`SPC26CD` elsewhere in this catalogue -- kept as `lhb_code`
- `code` (e.g. `7A6`) -- NHS ODS organisational code, a different coding system used by NHS Wales' own systems -- kept alongside as `lhb_ods_code`, not dropped
- `source`, `capturedby`, `capt_date` -- identical across all 7 rows (single statutory instrument, one digitisation batch) -- provenance facts, not per-row data, so dropped from the table and recorded here instead: legal basis is NHS Wales Statutory Instrument 2009 No. 778 (as amended), digitised by Welsh Government Cartographics, captured 2019-04-01.


In [6]:
wfs_url = (
    "https://datamap.gov.wales/geoserver/wfs"
    "?service=WFS&version=2.0.0&request=GetFeature"
    "&typeName=geonode:lhb_high_water_mark_post_apr_2019"
    "&outputFormat=application/json"
)

lhb = gpd.read_file(wfs_url)
print(len(lhb))
print(lhb.columns)

7
Index(['id', 'fid', 'area_code', 'code', 'name_en', 'name_cy', 'area_descr',
       'source', 'capturedby', 'capt_date', 'geometry'],
      dtype='str')


In [9]:
lhb = lhb.drop(columns=["id", "fid", "source", "capturedby", "capt_date"])

column_rename = {
    "area_code":  "lhb_code",       # ONS-style GSS code, e.g. W11000028
    "code":       "lhb_ods_code",   # NHS ODS organisational code, e.g. 7A6
    "name_en":    "lhb_name",
    "name_cy":    "lhb_name_welsh",
    "area_descr": "area_type",
}

lhb = lhb.rename(columns=column_rename)
lhb.head()

,lhb_code,lhb_ods_code,lhb_name,lhb_name_welsh,area_type,geometry
0,W11000028,7A6,Aneurin Bevan University Health Board,Bwrdd Iechyd Prifysgol Aneurin Bevan,Local Health Board,"MULTIPOLYGON (((326793.177 232168.977, 326793...."
1,W11000029,7A4,Cardiff and Vale University Health Board,Bwrdd Iechyd Prifysgol Caerdydd a'r Fro,Local Health Board,"MULTIPOLYGON (((324948.695 178653.899, 324948...."
2,W11000030,7A5,Cwm Taf Morgannwg University Health Board,Bwrdd Iechyd Prifysgol Cwm Taf Morgannwg,Local Health Board,"MULTIPOLYGON (((287613.451 176537.676, 287613...."
3,W11000031,7A3,Swansea Bay University Health Board,Bwrdd Iechyd Prifysgol Bae Abertawe,Local Health Board,"MULTIPOLYGON (((257560.054 201439.298, 257560...."
4,W11000025,7A2,Hywel Dda University Health Board,Bwrdd Iechyd Prifysgol Hywel Dda,Local Health Board,"MULTIPOLYGON (((257560.053 201439.292, 257560...."


In [10]:
lhb.to_file(icb_path, driver="GPKG", layer="lhb_2019_WA", index=False)

## Scotland

## NHS Health Boards (Scotland, 2014 boundaries)

**Note on source authority:** the URL originally supplied in the github issue (`services-eu1.arcgis.com/cECIr59LclpO818r/...`) traces to a Stirling Council open-data repost, not the primary source. The actual authority is the Scottish Government's own GI-SAT team. National Records of Scotland explicitly does **not** hold this geography -- NRS's own "geographies we do not hold" page points to SpatialData.gov.scot / Scottish Government instead. Worth remembering for any future Scottish layers in this catalogue: NRS is not the default authority for everything Scottish the way ONS is for England/UK-wide layers.

**Note on legal/currency basis:** boundaries defined by the National Health Service (Variation of Areas of Health Boards) (Scotland) Order 2013 (SSI 2013/347), in force since 1 April 2014. Unchanged since -- 14 regional Health Boards (there are also 6-7 non-geographic Special Boards, e.g. Scottish Ambulance Service, NHS 24, which this layer does not cover, since it's boundaries-only).

**Note -- two ScotGov hosts, only one usable:**
- `maps.gov.scot/server/...` -- metadata/schema pages load fine, but `/query` returns the Esri REST Directory HTML page instead of JSON, even with correct parameters. The "Login | Get Token" banner suggests query operations are gated behind auth on this internal-facing host, even though the layer itself is meant to be open data.
- `maps.data.gov.scot/arcgis/...` -- the actual public open-data mirror of the same layer. This is the one that works, but needed several fixes to get there:

**Fix 1 -- bot detection:** first request came back blocked outright.
Adding a browser-like `User-Agent` header got past it:

```python
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
```

**Fix 2 -- `outFields=*` breaks on this layer:** once past bot detection, `outFields=*` consistently 500'd (`"Error performing query operation"`). Root cause: the schema includes computed fields with parentheses in their names -- `st_area(shape)` / `st_perimeter(shape)` -- which this server's `outFields=*` expansion can't handle. Fix: name real fields explicitly (`objectid,hbcode,hbname`) instead of using `*`.

**Fix 3 -- geometry batch limit:** even with explicit fields, any request with `returnGeometry=true` for all 14 boards at once still 500'd -- regardless of `f=json` or `f=geojson`. Tested each board individually by `objectid` and all 14 succeeded on their own, which ruled out a corrupt/malformed geometry and pointed instead at a response-size or complexity limit specific to this server's batch geometry serialization. Worked around by fetching geometry one record at a time and concatenating.

In [28]:

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
base_url = "https://maps.data.gov.scot/arcgis/rest/services/NHSHealthBoards/MapServer/0/query"

# Get the list of objectids first
params_ids = {"where": "1=1", "outFields": "objectid,hbcode,hbname", "returnGeometry": "false", "f": "json"}
r = requests.get(base_url, params=params_ids, headers=headers, timeout=60)
ids = [f["attributes"]["objectid"] for f in r.json()["features"]]

# Fetch geometry one record at a time and combine
frames = []
for oid in ids:
    params = {
        "where": f"objectid={oid}",
        "outFields": "objectid,hbcode,hbname",
        "returnGeometry": "true",
        "f": "geojson",
    }
    r = requests.get(base_url, params=params, headers=headers, timeout=60)
    data = r.json()
    frames.append(gpd.GeoDataFrame.from_features(data["features"]))

hb_scotland = pd.concat(frames, ignore_index=True)
print(len(hb_scotland))
print(hb_scotland.columns)

14
Index(['geometry', 'objectid', 'hbcode', 'hbname'], dtype='str')


In [29]:
hb_scotland = hb_scotland.drop(columns=["objectid"])  # internal row ID, not a real attribute

column_rename = {
    "hbcode": "hb_code",   # e.g. S08000015 -- ONS-style GSS code
    "hbname": "hb_name",
}

hb_scotland = hb_scotland.rename(columns=column_rename)
hb_scotland.head()

,geometry,hb_code,hb_name
0,"MULTIPOLYGON (((-4.79896 55.88942, -4.79899 55...",S08000015,Ayrshire and Arran
1,"MULTIPOLYGON (((-2.36673 55.94597, -2.36671 55...",S08000016,Borders
2,"MULTIPOLYGON (((-3.97443 55.4579, -3.97468 55....",S08000017,Dumfries and Galloway
3,"MULTIPOLYGON (((-4.33441 56.53455, -4.33426 56...",S08000019,Forth Valley
4,"MULTIPOLYGON (((-3.33696 57.7248, -3.33728 57....",S08000020,Grampian


In [30]:
hb_scotland[["hb_code","hb_name","geometry"]].to_file(icb_path, driver="GPKG", layer="hb_2014_SC", index=False)

c:\Users\spspa\miniforge3\envs\raster_env\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


## Northern Ireland

## Health and Social Care Trusts (Northern Ireland, 1993 base / 2017 review)

**Note on dates -- three that mean different things:**
- **1993** -- the source geometry is derived from Land and Property  Services' (LPS) 50K-scale Local Government Districts, as they existed  in 1993
- **2017** -- DHSSPS/Department of Health completed a review of the original Trust Boundary dataset, removing scale variations and surplus attribute information introduced by the original 1:50K digitisation process
- **2018** (21/02/2018) -- this specific file's last publication date on Open Data NI

None of these dates mean the current 5 Trusts are outdated -- NI's HSC Trusts (Belfast, Northern, Southern, South Eastern, Western) haven't been reorganised since this 2017 review; "Frequency of Update" on the dataset page is explicitly listed as "Not Planned."

**Note on access:** `www.opendatani.gov.uk` blocks automated fetches(`robots.txt` disallow). The mirror at `admin.opendatani.gov.uk` allowsdirect page fetches but is JS-rendered, so the actual resource downloadlinks aren't in the raw HTML -- and CKAN's standard `/api/3/action/package_show`endpoint, which would normally list them programmatically, returnedforbidden here. Resolved by downloading the GeoJSON manually via browserinstead of scripting the fetch -- reasonable given this is a static,rarely-updated file, not something that needs repeat automated pulls.

In [ ]:
# Check for an available source. 

r = requests.get("https://admin.opendatani.gov.uk/api/3/action/package_show",
                  params={"id": "department-of-health-trust-boundaries"}, timeout=30)
print(r.status_code)
resources = r.json()["result"]["resources"]
for res in resources:
    print(res["format"], "-", res["url"])

200
SHP - https://admin.opendatani.gov.uk/dataset/0b04b46c-49af-45d5-b277-91b10937a01b/resource/7fa52dde-90b8-446e-bb79-4871d1028cb4/download/dohtrustboundary.zip
GeoJSON - https://admin.opendatani.gov.uk/dataset/0b04b46c-49af-45d5-b277-91b10937a01b/resource/645f8eef-8813-47a9-bb1e-a4932ada721a/download/trustboundaries.geojson


In [34]:
hsc_trusts = gpd.read_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\raw\trustboundaries.geojson")  # url from the resources list above
print(len(hsc_trusts))  # expect 5
print(hsc_trusts.columns)

5
Index(['TrustName', 'TrustCode', 'Shape_Leng', 'Shape_Area', 'geometry'], dtype='str')


In [36]:
hsc_trusts = hsc_trusts.drop(columns=["Shape_Leng", "Shape_Area"])  # OGR-computed, not source attributes

column_rename = {
    "TrustName": "trust_name",
    "TrustCode": "trust_code",
}

hsc_trusts = hsc_trusts.rename(columns=column_rename)
hsc_trusts.head()

,trust_name,trust_code,geometry
0,Belfast Health and Social Care Trust,BHSCT,"MULTIPOLYGON (((-5.92365 54.65013, -5.92324 54..."
1,Northern Health and Social Care Trust,NHSCT,"MULTIPOLYGON (((-5.82152 54.87545, -5.82152 54..."
2,Western Health and Social Care Trust,WHSCT,"MULTIPOLYGON (((-6.96342 55.19494, -6.96334 55..."
3,Southern Health and Social Care Trust,SHSCT,"MULTIPOLYGON (((-6.27718 54.5387, -6.27736 54...."
4,South Eastern Health and Social Care Trust,SEHSCT,"MULTIPOLYGON (((-5.54494 54.29734, -5.54494 54..."


In [37]:
hsc_trusts.to_file(icb_path, driver="GPKG", layer="hsc_trusts_2018_NI", index=False)


**File complete:** `health_boards.gpkg` now holds all four nations' health-board-equivalent geographies as separate, country-suffixed layers -- `sub_icb_apr2026_EN`, `lhb_2019_WA`, `hb_2014_SC`, `hsc_trusts_2018_NI` -- reflecting that these are genuinely different kinds of geography per nation (ICBs, Local Health Boards, NHS Health Boards, HSC Trusts), not a single UK-wide product split by country the 
way Westminster constituencies were.